# Create Parkinsonfonden Awards (Sweden)

Creates Parkinsonfonden awards from the fund's own "Beviljade anslag" page,
https://www.parkinsonfonden.se/forskning/beviljade-anslag/ : one block per
grant ("<Name>, <Institution>, <amount> kronor" + project title or trip
purpose), grouped by call. The live page (2026 layout) shows 2021-2026; the
previous layout of the same page listed the full history 2010-2024 and is read
from the Internet Archive capture of the fund's page (20241210014650, raw `id_`
capture). Rows on both are deduplicated on (kind, year, name, amount).
**848 awards, 2010-2026, SEK 138M** (local run 2026-10-01): 528 project
grants, 313 travel grants, 7 equipment grants.

**Scope:** project, equipment (apparatur) and travel grants to Parkinson's
researchers; all kept (travel = `funding_type travel`, flagged for review). The
fund's two prizes (Åke Ljungdahl, Elsa och Inge Andersson) are separate pages
and not included. SweCRIS does not carry Parkinsonfonden (scan 2026-10-01).

**`funder_award_id` (§2.1.1):** citing works write the fund's application
numbers (`682/14`, `1115/18`, `1443/2022`); the page prints them only for the
October 2014 call (`Nr 682/14, ...`), so those 39 rows ship that number. All
other rows ship a synthetic `PF-<P|R|A><year>-<family>-<given>` key (P project,
R travel, A equipment; `-2` for a second distinct grant to the same person in
one call). Of the 77 existing citation stubs for F4320332679, 2 collapse
(663/14, 682/14). Twin note: ParkinsonFörbundet (F4320310526, the patients'
association) carries 3 stubs, one of them a Parkinsonfonden number (703/14).

**Dates / amounts:** `start_year` = the call year printed in the section
heading, `start_date` NULL. Amount in SEK as printed (one year's allocation;
multi-year projects reappear per year). The 2010 travel list gives names only
(16 rows, amount NULL).

**Prerequisites:** run `scripts/local/parkinsonfonden_to_s3.py` first. It uploads
`s3://openalex-ingest/awards/parkinsonfonden/parkinsonfonden_projects.parquet`
(§1.4 shrink guard).

**Funder (Path A, F4320* Crossref-registered):** funder_id `4320332679`,
Parkinsonfonden (ror 0300wre24, doi 10.13039/100008444), read from
`openalex.funders.funders` by id.

**Priority:** `581` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.parkinsonfonden_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/parkinsonfonden/parkinsonfonden_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_rows, COUNT(DISTINCT funder_award_id) as distinct_ids
FROM openalex.awards.parkinsonfonden_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.parkinsonfonden_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.parkinsonfonden_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320332679 must resolve to exactly 1 row in `openalex.funders.funders`. If 0 rows, STOP and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320332679;

## Step 2: Create Parkinsonfonden Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.parkinsonfonden_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320332679  -- Parkinsonfonden
),
src AS (
    SELECT
        TRIM(funder_award_id) AS funder_award_id,
        kind,
        COALESCE(NULLIF(TRIM(title), ''), NULLIF(TRIM(purpose), '')) AS title,
        TRY_CAST(amount AS DOUBLE) AS amount,
        TRY_CAST(call_year AS INT) AS call_year,
        NULLIF(TRIM(lead_given_name), '') AS given_name,
        NULLIF(TRIM(lead_family_name), '') AS family_name,
        NULLIF(TRIM(co_lead_given_name), '') AS co_given_name,
        NULLIF(TRIM(co_lead_family_name), '') AS co_family_name,
        NULLIF(TRIM(institution), '') AS institution,
        source_url
    FROM openalex.awards.parkinsonfonden_raw
    WHERE funder_award_id IS NOT NULL AND TRIM(funder_award_id) != ''
)
SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,
    g.title as display_name,
    CAST(NULL AS STRING) as description,
    f.funder_id,
    g.funder_award_id,
    g.amount,
    CASE WHEN g.amount IS NOT NULL THEN 'SEK' END as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    CASE g.kind WHEN 'travel' THEN 'travel' WHEN 'equipment' THEN 'infrastructure' ELSE 'research' END as funding_type,
    CASE g.kind WHEN 'travel' THEN 'Reseanslag' WHEN 'equipment' THEN 'Apparaturanslag' ELSE 'Projektanslag' END as funder_scheme,
    'parkinsonfonden_beviljade_anslag' as provenance,
    CAST(NULL AS DATE) as start_date,
    CAST(NULL AS DATE) as end_date,
    g.call_year as start_year,
    CAST(NULL AS INT) as end_year,
    CASE WHEN g.family_name IS NOT NULL THEN named_struct(
        'given_name', g.given_name,
        'family_name', g.family_name,
        'orcid', CAST(NULL AS STRING),
        'role_start', CAST(NULL AS DATE),
        'affiliation', named_struct(
            'name', g.institution,
            'country', CAST(NULL AS STRING),
            'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
        )
    ) END as lead_investigator,
    CASE WHEN g.co_family_name IS NOT NULL THEN named_struct(
        'given_name', g.co_given_name,
        'family_name', g.co_family_name,
        'orcid', CAST(NULL AS STRING),
        'role_start', CAST(NULL AS DATE),
        'affiliation', named_struct(
            'name', g.institution,
            'country', CAST(NULL AS STRING),
            'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
        )
    ) END as co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) as investigators,
    g.source_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM src g
CROSS JOIN src_funder f;

In [ ]:
%sql
-- §2.1.1 shape check: 2014 application numbers (682/14) or synthetic PF- keys; expect 0 rows.
SELECT funder_award_id, display_name
FROM openalex.awards.parkinsonfonden_awards
WHERE NOT (funder_award_id RLIKE '^[0-9]{3,4}/14$'
        OR funder_award_id RLIKE '^PF-[PRA][0-9]{4}-[a-z0-9-]+$')
LIMIT 20;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'parkinsonfonden_beviljade_anslag' AND priority = 581;

-- Priority 581: direct-from-funder ingest of Parkinsonfonden's grant page.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    581 as priority
FROM openalex.awards.parkinsonfonden_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.parkinsonfonden_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, amount, currency, start_year,
       lead_investigator.given_name, lead_investigator.family_name, lead_investigator.affiliation.name
FROM openalex.awards.parkinsonfonden_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, start_year, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_sek
FROM openalex.awards.parkinsonfonden_awards
GROUP BY funder_scheme, start_year ORDER BY start_year, funder_scheme;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.parkinsonfonden_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.parkinsonfonden_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.7 coverage (expect ~100% title/PI, ~98% amount).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) as pct_amount,
    collect_set(currency) as currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_year) as has_start_year,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.parkinsonfonden_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'parkinsonfonden_beviljade_anslag'
GROUP BY provenance, priority;